In [ ]:
import pandas as pd

In [ ]:
q12024 = pd.read_excel('2024comparendosQ1.xlsx', header=9)
q22024 = pd.read_excel('2024comparendosQ2.xlsx', header=9)
q32024 = pd.read_excel('2024comparendosQ3.xlsx', header=9)
q42024 = pd.read_excel('2024comparendosQ4.xlsx', header=9)

In [ ]:
q12024['TRIMESTRE'] = 1
q22024['TRIMESTRE'] = 2
q32024['TRIMESTRE'] = 3
q42024['TRIMESTRE'] = 4

In [ ]:
df2024total = pd.concat(
    [q12024, q22024, q32024, q42024],
    ignore_index=True
)

In [ ]:
poblacion2024 = pd.read_excel('Poblacion2024TotalMunicipios.xlsx')

cols = ['CODIGO_DIVIPOLA', 'Departamento', 'MUNICIPIO', 'PoblacionTotal2024']
dfmerge = poblacion2024[cols].copy()

dfmerge['CODIGO_DIVIPOLA'] = (
    pd.to_numeric(dfmerge['CODIGO_DIVIPOLA'], errors='coerce')
    .astype('Int64')
)
df2024total['CODIGO_DIVIPOLA'] = (
    pd.to_numeric(df2024total['CODIGO_DIVIPOLA'], errors='coerce')
    .astype('Int64')
)

df2024total = df2024total.merge(
    dfmerge,
    on='CODIGO_DIVIPOLA',
    how='left',
    suffixes=('', '_pob')
)

df2024total['DEPARTAMENTO'] = (
    df2024total['Departamento']
    .fillna(df2024total['DEPARTAMENTO'])
)
df2024total['MUNICIPIO'] = (
    df2024total['MUNICIPIO_pob']
    .fillna(df2024total['MUNICIPIO'])
)

df2024total = df2024total.rename(
    columns={'PoblacionTotal2024': 'PoblacionTotalMunicipio2024'}
)
df2024total = df2024total.drop(
    columns=['Departamento', 'MUNICIPIO_pob']
)

In [ ]:
municipio2024 = (
    df2024total
    .groupby(
        ['CODIGO_DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO'],
        as_index=False
    )
    .agg(
        comparendos=('CANTIDAD', 'sum'),
        poblacion=('PoblacionTotalMunicipio2024', 'max')
    )
)

municipio2024['poblacion'] = (
    municipio2024['poblacion'].replace(0, pd.NA)
)
municipio2024['tasacomparendos100k'] = (
    municipio2024['comparendos'] / municipio2024['poblacion'] * 100000
)

In [ ]:
df2024total.to_csv('comparendos2024.csv', index=False)